# MOTIF stratified subsample extraction

Extracts the frozen, seeded, per-family subsample of disarmed PE files from
[boozallen/MOTIF](https://github.com/boozallen/MOTIF), per
[ADR-0020](https://github.com/Tanishk75/MalMap/blob/master/docs/adr/0020-motif-replaces-malimg-second-track.md)
and the frozen protocol at
[`protocols/motif_sampling.md`](https://github.com/Tanishk75/MalMap/blob/master/protocols/motif_sampling.md).

**Why on Kaggle rather than locally.** MOTIF's files carry known-malware MD5 hashes even though
they are disarmed and cannot execute. A managed endpoint antivirus (observed: Trend Micro Apex
One, not Windows Defender, running on the development machine and not locally excludable) deletes
them on sight the moment they land on local disk as plain files. Running the extraction here
avoids that entirely during selection and verification; step 7 then re-packages the 630 selected
files into a **password-protected 7z** (the same trick MOTIF's own upstream distribution already
uses) so the download back to a local, AV-managed machine also survives -- the archive sits
encrypted at rest and is only decompressed on demand by pipeline code that supplies the password.

**Before running this notebook:** confirm `protocols/motif_sampling.md` in the repo is still the
version you intend to run against -- this notebook enforces it, it does not decide it. If different
numbers are needed, edit that file in the repo, commit it, and re-run this notebook against the
new commit.

**What this notebook does NOT do:** extract anything outside the frozen 630-file selection, or
silently change the sampling protocol if live counts disagree with it.

## 1. Clone the repo and install dependencies
Single source of truth for the sampling constants, the selection procedure and the
registry-building code lives in the repo (`src/`, `scripts/`), not duplicated in this notebook.

In [ ]:
!git clone --depth 1 https://github.com/Tanishk75/MalMap.git /kaggle/working/MalMap

import sys
sys.path.insert(0, "/kaggle/working/MalMap")

!pip install -q py7zr

## 2. Clone the MOTIF dataset repo
Public, no request gate (ADR-0020) -- unlike BIG2015, this needs no Kaggle "Add Data" attachment.
`MOTIF.7z` is stored in the repo via Git LFS.

In [ ]:
!git lfs install
!git clone --depth 1 https://github.com/boozallen/MOTIF.git /kaggle/working/MOTIF
!cd /kaggle/working/MOTIF && git lfs pull --include=MOTIF.7z

from pathlib import Path

MOTIF_REPO = Path("/kaggle/working/MOTIF")
MOTIF_DATASET_JSONL = MOTIF_REPO / "dataset" / "motif_dataset.jsonl"
MOTIF_7Z = MOTIF_REPO / "MOTIF.7z"

assert MOTIF_DATASET_JSONL.exists(), f"{MOTIF_DATASET_JSONL} missing -- clone did not complete"
assert MOTIF_7Z.exists() and MOTIF_7Z.stat().st_size > 10**6, (
    f"{MOTIF_7Z} missing or too small -- git lfs pull did not fetch the real file"
)
print("MOTIF.7z size (MB):", MOTIF_7Z.stat().st_size / 1024**2)

## 3. Verify published per-family counts against protocols/motif_sampling.md
The protocol records the top-30 counts read from the dataset's own metadata at the time it was
frozen. This cell recomputes them from the freshly cloned motif_dataset.jsonl and stops if they
disagree -- a silent mismatch here would make the Development Plan's "extracted set matches the
protocol exactly" exit-gate check meaningless.

In [ ]:
import json as jsonlib
import pandas as pd

EXPECTED_COUNTS = {
    "icedid": 142, "azorult": 68, "phorpiex": 58, "maze": 52, "trickbot": 43,
    "gandcrab": 41, "locky": 41, "artradownloader": 40, "redaman": 40, "seduploader": 40,
    "egregor": 37, "prometei": 35, "shamoon": 33, "zegost": 32, "indigodrop": 31,
    "peppyrat": 29, "turnedup": 29, "ryuk": 28, "medusalocker": 27, "mosaicregressor": 27,
    "crat": 26, "valak": 26, "bazarbackdoor": 25, "olympicdestroyer": 25, "wannacry": 25,
    "copperhedge": 22, "andromeda": 21, "dreambot": 21, "loda": 21, "ursnif": 21,
}

records = []
with open(MOTIF_DATASET_JSONL, encoding="utf-8") as f:
    for line in f:
        rec = jsonlib.loads(line)
        records.append({"md5": rec["md5"], "family_label": rec["reported_family"]})
df = pd.DataFrame(records)

live_counts = df["family_label"].value_counts().to_dict()
mismatches = {
    fam: (EXPECTED_COUNTS[fam], live_counts.get(fam))
    for fam in EXPECTED_COUNTS
    if EXPECTED_COUNTS[fam] != live_counts.get(fam)
}

print(pd.DataFrame({"expected": EXPECTED_COUNTS,
                     "live": {fam: live_counts.get(fam) for fam in EXPECTED_COUNTS}}))

if mismatches:
    raise AssertionError(
        f"live per-family counts disagree with protocols/motif_sampling.md: {mismatches}. "
        "Fix the protocol file in the repo first -- do not proceed with a stale table."
    )
print("Live counts match the frozen protocol.")

## 4. Select the frozen sample list
Reuses select_frozen_sample from scripts/build_motif_subsample.py -- the exact procedure
protocols/motif_sampling.md describes (sorted family order, sorted md5s within a family, one
RandomState stream seeded once, first min(cap, available) after shuffling) lives in one place,
not copied into this notebook.

In [ ]:
from scripts.build_motif_subsample import (
    MOTIF_PASSWORD,
    select_frozen_sample,
    extract_selected,
    verify_pefile_parses,
)
from src.config import MOTIF_SAMPLE_SEED, MOTIF_SAMPLES_PER_FAMILY, MOTIF_TOP_N_FAMILIES
from src.data.registry import build_registry, make_splits

print("seed:", MOTIF_SAMPLE_SEED, "top families:", MOTIF_TOP_N_FAMILIES,
      "cap per family:", MOTIF_SAMPLES_PER_FAMILY)

selected = select_frozen_sample(MOTIF_DATASET_JSONL)
print(f"Selected {len(selected)} samples across {selected['family_label'].nunique()} families")
print(selected["family_label"].value_counts())

## 5. Extract exactly the selected 630 files from the password-protected archive
Targeted extraction by exact filename match against the frozen selection -- never a scan-and-guess
(ADR-0020). Runs on Kaggle's disk, not the local machine, so there is nothing here yet for a local
antivirus to react to.

In [ ]:
OUT_DIR = Path("/kaggle/working/motif_subsample")
BYTES_DIR = OUT_DIR / "bytes"

extract_selected(MOTIF_7Z, selected, BYTES_DIR)
extracted = sorted(p.name for p in BYTES_DIR.glob("MOTIF_*"))
assert len(extracted) == len(selected), f"extracted {len(extracted)}, expected {len(selected)}"
print(f"Extracted {len(extracted)} files to {BYTES_DIR}")

## 6. Write motif_labels.csv, build the registry, verify pefile parses
Matches the layout src.data.registry.build_registry("motif", ...) expects. The pefile check is
Milestone 0's exit gate (ADR-0020): confirms at least one MOTIF sample parses as a PE before FR6
relies on it, falling back to benign binaries (ADR-0018) if the parse rate is too low.

In [ ]:
labels_path = OUT_DIR / "motif_labels.csv"
selected.to_csv(labels_path, index=False)
print(f"Wrote {len(selected)} rows to {labels_path}")

registry = build_registry("motif", str(OUT_DIR))
registry = make_splits(registry)
print(registry["family_label"].value_counts())
print(registry["split"].value_counts())

check_ids = selected["md5"].tolist()[:20]
parsed, total = verify_pefile_parses(BYTES_DIR, check_ids)
print(f"pefile parsed {parsed}/{total} sampled files")
if parsed == 0:
    print(
        "WARNING: pefile parsed none of the sampled files. FR6 falls back to the "
        "benign-binary mechanism check (ADR-0018) per ADR-0020's contingency."
    )

## 7. Copy the registry and family map, then re-package as a password-protected 7z
build_registry/make_splits persist under the cloned repo's data_cache/ by default; copy them
alongside the extracted files first so the export is self-contained, exactly as the BIG2015
notebook does. Then, unlike BIG2015 (whose .bytes files are a scrubbed, non-executable format
with no malware signature to trip a scanner), wrap the whole thing in a password-protected archive
with encrypted headers -- so filenames and content are both opaque to any antivirus scanning the
download, until pipeline code decompresses it on purpose with the password.

In [ ]:
import shutil
import py7zr

repo_data_cache = Path("/kaggle/working/MalMap/data_cache")
shutil.copy(repo_data_cache / "registry_motif.csv", OUT_DIR / "registry_motif.csv")
shutil.copy(repo_data_cache / "family_to_id_motif.json", OUT_DIR / "family_to_id_motif.json")

print("Pre-archive export layout:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and p.parent == OUT_DIR:
        print(" ", p.relative_to(OUT_DIR))
print(f"  bytes/  ({len(list(BYTES_DIR.glob('MOTIF_*')))} files)")

ARCHIVE_PATH = Path("/kaggle/working/motif_subsample.7z")
with py7zr.SevenZipFile(ARCHIVE_PATH, "w", password=MOTIF_PASSWORD,
                         header_encryption=True) as archive:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            archive.write(p, arcname=str(p.relative_to(OUT_DIR)))

print(f"Wrote {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024**2:.1f} MB), "
      "password-protected with header encryption.")

## 8. Export before the session ends
Per ADR-0006, nothing under /kaggle/working survives past the session on its own. Use
"Save Version" with output files enabled, or download motif_subsample.7z directly from the
notebook's output pane.

**Local use.** Unpack with py7zr (or 7-Zip, entering the password) directly into
data/raw/motif/, so the layout is data/raw/motif/bytes/MOTIF_<md5> plus
data/raw/motif/motif_labels.csv, registry_motif.csv, family_to_id_motif.json -- matching what
build_registry("motif", "data/raw/motif") expects. If a managed antivirus on the local machine is
still an issue, decompress just before running the pipeline stage that needs the raw bytes and
avoid leaving the plain files sitting on disk longer than necessary.